# LangChain + Clembench experiment constructor

**Single configurable notebook** for running LangChain agents on clembench games.

**Only Cell 2 (Config) needs to be changed between experiments.**

| Variable | Description |
|---|---|
| `GAME` | any clembench game (e.g., `taboo`, `referencegame`, etc.|
| `MODEL` | any model (e.g.,`qwen`, `gpt-4o-mini`) |
| `AGENT_TYPE` | a LangChain agent (choose from below or create a custom one) |
| `RUN_ID` | any string — used as the results folder name |
| `NUM_EPISODES` | integer |
| `SINGLE_PASS` | `True` = one pass through instances, `False` = cycle infinitely |

In [307]:
# ── CONFIG ──────────────────────────────────────────────────────────
GAME         = "taboo"   
MODEL        = "clp-chat"            
AGENT_TYPE   = "LongTermPlanningAgent"  
RUN_ID       = "infinite"
NUM_EPISODES = 15
SINGLE_PASS  = True
# ───────────────────────────────────────────────────────────────────────

## 1. Preparation

In [308]:
import os

CLEMBENCH_HOME = r"C:\Users\white\Desktop\agents_experiments\clembench_v3"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [309]:
# uncomment and run once to install dependencies, then comment out again
# %pip install -r $CLEMBENCH_HOME/requirements.txt
# %pip install --upgrade ipywidgets jupyter_client clemcore

In [310]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s {GAME}

clem 3.5.0
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'taboo'}'
taboo:
 	Taboo game between two agents where one has to describe a word for
	the other to guess.


In [311]:
import json

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from playpen.agents import ClemAgent, ClemObservation
from clemcore.clemgame import env, episode_results_folder_callbacks

from clemcore.backends import ModelRegistry
from clemcore.backends import KeyRegistry

In [312]:
#register the model if necessary

#registry = ModelRegistry.register("insert model name", backend="insert backend",
 #                                 model_id="insert model id")
#registry.get_first_model_spec_that_unify_with("insert model id")

In [313]:
#register the API key and url if necessary

#API_KEY = ""
#ORGANIZATION = ""
#BASE_URL ="" 

API_KEY = "asd-asd-asd"
ORGANIZATION = "<insert-organisation-here>"
BASE_URL ="https://jarvis.ling.uni-potsdam.de/clp-chat/api/v1/" 
KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)
#KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

KeyRegistry(file='C:\Users\white\Desktop\agents_experiments\notebooks\key.json', backends=[openai_compatible])

In [314]:
import httpx

client = httpx.Client(verify=False)

def create_model(
    model_name: str,
    registry_path: str = "model_registry.json",
    key_path: str = "key.json",
    temperature: float = 0,
    max_tokens: int = 300) -> ChatOpenAI:
    """Returns a configured ChatOpenAI instance by looking up model_name in model_registry.json.
    Raises ValueError if the model or its required backend credentials are not found.
    """
    with open(registry_path) as f:
        registry = json.load(f)

    entry = next((e for e in registry if e["model_name"] == model_name), None)
    if entry is None:
        raise ValueError(f"Model {model_name!r} not found in {registry_path}.")

    backend = entry["backend"]

    with open(key_path) as f:
        keys = json.load(f)

    if backend not in keys:
        raise ValueError(
            f"Backend {backend!r} (required by model {model_name!r}) "
            f"not found in {key_path}."
        )

    credentials = keys[backend]

    return ChatOpenAI(
        model=entry["model_id"],
        base_url=credentials["base_url"],
        api_key=credentials["api_key"],
        temperature=temperature,
        max_tokens=max_tokens,
        http_client=client
    )

## 2. Agent constructor

In [315]:
# Subagent prompt used by MyAgenticPlayer 
_SUBAGENT_PROMPT = """
You are given a small piece of text which contains gameplay rules. You need to extract the necessary tags
(often written in CAPITAL LETTERS), so that the player can use them for the answer.
Do not output any text apart from the tag(s). Example IO pair:

INPUT:
Let's play a guessing game! Your task is to answer the other player's questions. Based on your knowledge
of the word: $TARGET WORD$, respond to the following questions or guesses. Limit your response to only
'yes' or 'no' with no explanation or other words. Never reveal the answer in your response.

You must reply using the format below and DO NOT ADD ANY TEXT OTHER THAN THIS:

ANSWER: <some text>

Target Word: $TARGET WORD$

OUTPUT:
ANSWER:

If you identified no tags, please return NO TAG as an answer.
"""


# Agent definitions 

class CoreToolsAgent(ClemAgent):
    """Agent with remember / recall / observe / get_observations tools."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._remember_rules(),
            self._recall_rules(),
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have memory tools to help you play any game effectively.

  STRATEGY:follow this order every turn:

  TURN 1 (rules turn):
    1. Store the goal, required response format, and constraints using remember_rules().
    2. Give your first response in the required format.

  TURN 2+ (every subsequent turn):
    1. Call observe_game() to record the new information you just received.
    2. Call get_game_observations() to review what has already happened.
    3. Call recall_rules() to keep to the game rules.
    4. Give your response based on the full picture.

  Never skip steps 1-2 on turn 2+. Observations are required before acting."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _remember_rules(self):
        store = self.store
        @tool
        def remember_rules(key: str, value: str) -> str:
            """
            Store any important information concerning the game rules.

            Examples:
                remember_rules("goal", "describe the target without forbidden words")
                remember_rules("format", "CLUE: <text>")
                remember_rules("target", "first grid")
                remember_rules("forbidden", "cat, dog, pet")
            """
            store[key] = value
            return f"Stored: {key} = {value}"
        return remember_rules

    def _recall_rules(self):
        store = self.store
        @tool
        def recall_rules(key: str = "") -> str:
            """
            Retrieve stored information about the game rules.

            Args:
                key: Specific key, or empty for everything
            """
            if not store:
                return "Memory empty."
            if key and key in store:
                return f"{key}: {store[key]}"
            return "\n".join(f"- {k}: {v}" for k, v in store.items())
        return recall_rules

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue mentions 'round shape'")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"


class MyAgenticPlayer(ClemAgent):
    """Agent that uses an extract_tags subagent to parse game rules on first turn."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.model = model
        self.memory = InMemorySaver()
        self.base_thread_id = thread_id
        self.episode = 0

        @tool
        def extract_tags(initial_prompt: str) -> str:
            """Extract the response tags that are necessary for the player from the game rules."""
            subagent = create_agent(model=self.model, tools=[], system_prompt=_SUBAGENT_PROMPT)
            result = subagent.invoke({"messages": [{"role": "user", "content": initial_prompt}]})
            final = next(m for m in reversed(result["messages"]) if isinstance(m, AIMessage))
            return final.content

        self.agent = create_agent(
            model=self.model,
            tools=[extract_tags],
            checkpointer=self.memory,
            system_prompt=(
                "You're going to play a game. You're a professional agent game player. "
                "You have a helpful tool extract_tags that identifies the required response format. "
                "On your first turn, call extract_tags and use the result to format all future responses."
            ),
        )

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}},
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"


class TwoToolsAgent(ClemAgent):
    """Agent with observe / get_observations tools."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have memory tools to help you play any game effectively.

  STRATEGY:follow this order every turn:

  TURN 1 (rules turn):
    Do not use any tools

  TURN 2+ (every subsequent turn):
    1. Call observe_game() to record the new information you just received.
    2. Call get_game_observations() to review what has already happened.
    You MUST call observe_game() and get_game_observations()  before you give the answer! Never skip steps 1-2 on turn 2+.."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue mentions 'round shape'")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [316]:

class LongTermPlanningAgent(ClemAgent):
    """Agent with long term memory for planning its actions."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self._tool_observations = []

        tools = [
            
            self._write_trajectory(),
            self._get_written_trajectories(),
        
        ]

        system_prompt = """You're a professional game player with planning tools.

    STRATEGY:

    On the FIRST turn: 
    1. Call write_trajectory() to save what you are about to answer.
    
    At the START of every subsequent episode:
    1. Call get_written_trajectories() to recall lessons from past episodes.
    2. Apply that knowledge to play better.

  At the END of every subsequent episode (just before you make your final move):
    1. Call write_trajectory() to save what you learned from the past interaction (was it successful or not?).
    Make your move in the format defined by the game rules. Do NOT output anything else!
    """

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def _write_trajectory(self):
        observations = self._tool_observations
        @tool
        def write_trajectory(observation: str) -> str:
            """
            Promptly reason about your performance througout the past rounds and make notes for yourself.
            Example: "tried Option 1, unsuccessful; game proceeds; need to observe the remaining options
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return write_trajectory

    def _get_written_trajectories(self):
        observations = self._tool_observations
        @tool
        def get_written_trajectories() -> str:
            """Get all strategies and trajectories you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations[-10:]))
        return get_written_trajectories
     

    def reset(self):
        super().reset()
        self.episode += 1


    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode.
        Stores messages of all types, the tool calls and their results."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        msgs = state.get("channel_values", {}).get("messages", [])
        return msgs[-10:]

    def get_longterm_snapshot(self) -> dict:
      """Return the long-term memory state (stores all observations made by the player
      throughout the whole game (not wiped between episodes)"""
      return {
          "observations": list(self._tool_observations),
      }

    def act(self, last: ClemObservation) -> str:
        
        memory_summary = ""
        
        if self._tool_observations:
          memory_summary = "STRATEGIES AND OBSERVATIONS FROM PAST EPISODES:\n"
          memory_summary += "Past observations:\n" + "\n".join(f"- {o}" for o in self._tool_observations[-10:])
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content + "\n\n" + memory_summary}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

## 3. Experiment setup

In [317]:
def make_agent(agent_type: str, model: ChatOpenAI, thread_id: str) -> ClemAgent:
    """Instantiate an agent by name."""
    if agent_type == "CoreToolsAgent":
        return CoreToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "MyAgenticPlayer":
        return MyAgenticPlayer(model=model, thread_id=thread_id)
    elif agent_type == "TwoToolsAgent":
        return TwoToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "LongTermPlanningAgent":
        return LongTermPlanningAgent(model=model, thread_id=thread_id)
    else:
        raise ValueError(f"Unknown agent type: {agent_type!r}.")

In [318]:
callbacks = episode_results_folder_callbacks(
    run_dir=RUN_ID,
    result_dir_path="playpen-records",
    player_model_infos=f"{AGENT_TYPE}-{MODEL}",
)

game_env = env(GAME, single_pass=SINGLE_PASS, callbacks=callbacks)
#removed reset

print("roles:", game_env.unwrapped.game_benchmark.game_spec["roles"])

2026-03-06 16:53:26,442 - clemcore.cli - INFO - Found '1' game matching the game_selector="taboo"
2026-03-06 16:53:26,444 - clemcore.cli - INFO - {
  "game_name": "taboo",
  "description": "Taboo game between two agents where one has to describe a word for the other to guess.",
  "main_game": "taboo",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0",
    "3.0"
  ],
  "regression": "small",
  "roles": [
    "Describer",
    "Guesser"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\agents_experiments\\clembench_v3\\taboo"
}
2026-03-06 16:53:26,446 - clemcore.run - INFO - Loading game benchmark for taboo
2026-03-06 16:53:26,454 - clemcore.run - INFO - Loading game benchmark for taboo took: 0:00:00.006999
2026-03-06 16:53:26,458 - clemcore.run - INFO - Prepared instance queue for taboo using 3 experiments ['high_en', 'medium_en', 'low_en'] and 15 instances in total.
2026-03-06 16:53:26,459 - clemcore.run - INFO - 

In [319]:
model = create_model(MODEL)

if GAME == "textmapworld":
    guesser = make_agent(AGENT_TYPE, model, thread_id="guesser")
    learner_agents = [guesser]
    agent_mapping = {"player_0": guesser, "player_1": None}  # player_1 refreshed in loop
else:
    describer = make_agent(AGENT_TYPE, model, thread_id="describer")
    guesser   = make_agent(AGENT_TYPE, model, thread_id="guesser")
    learner_agents = [describer, guesser]
    agent_mapping  = {"player_0": describer, "player_1": guesser}

print("Agent mapping:", {k: type(v).__name__ for k, v in agent_mapping.items()})

Agent mapping: {'player_0': 'LongTermPlanningAgent', 'player_1': 'LongTermPlanningAgent'}


## 4. Run game

In [320]:
import json as _json
from pathlib import Path
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


#TODO: make memory printing / saving OPTIONAL because not all planned setups need memory

def serialize_messages(msgs: list) -> list:
    """Convert LangChain messages to plain dicts for JSON serialization."""
    out = []
    for m in msgs:
        out.append({
            "type": type(m).__name__,
            "content": m.content,
            "tool_calls": getattr(m, "tool_calls", []),
        })
    return out

memory_log_dir = Path("playpen-records") / RUN_ID / GAME / "memory_logs"   #NB: kept separately from the results!
memory_log_dir.mkdir(parents=True, exist_ok=True)                   #create the directory if absent

all_episodes_data = []

for episode in range(NUM_EPISODES):
    game_env.reset()
    for agent in learner_agents:
        agent.reset()

    # textmapworld: built-in describer is recreated on every reset() — refresh the reference
    #TODO: smth has to be done about singleplayer games 
    if GAME == "textmapworld":
        agent_mapping["player_1"] = game_env.unwrapped.game_master.describer

    episode_memory_log = []  # collects memory snapshots for this episode

    context_response_pairs = []
    for step_idx, agent_id in enumerate(game_env.agent_iter()):
        context, reward, termination, truncation, info = game_env.last()
        response = None if (termination or truncation) else agent_mapping[agent_id](context)
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

        print(f"  [obs:{agent_id}] {context['content'][:100]!r}")

        # snapshot memory for the agent that just acted
        agent = agent_mapping.get(agent_id)
        if agent is not None and hasattr(agent, "get_memory_snapshot"):
            msgs = agent.get_memory_snapshot()
            snapshot = {
                "step": step_idx,
                "agent_id": agent_id,
                "thread_id": agent.base_thread_id,
                "messages": serialize_messages(msgs),
            }
            episode_memory_log.append(snapshot)
            print(f"  [memory:{agent.base_thread_id}] {len(msgs)} messages: "
                  + " | ".join(f"{type(m).__name__}({m.content[:40]!r})" for m in msgs))

    # write memory log for this episode 
    log_path = memory_log_dir / f"episode_{episode + 1:04d}.json"
    with open(log_path, "w") as f:
        _json.dump(episode_memory_log, f, indent=2, default=str)

    #the long-term memory
    for agent in learner_agents:
      if hasattr(agent, "get_longterm_snapshot"):
          lt_path = memory_log_dir / f"longterm_{agent.base_thread_id}.json"
          with open(lt_path, "w") as f:
              _json.dump(agent.get_longterm_snapshot(), f, indent=2, default=str)

    all_episodes_data.append(context_response_pairs)
    print(f"Episode {episode + 1}/{NUM_EPISODES} completed — {len(context_response_pairs)} steps")

print(f"\nAll episodes done. Memory logs written to: {memory_log_dir}")

  [obs:player_0] 'You are playing a collaborative word guessing game in which you have to describe a target word for a'
  [memory:describer] 4 messages: HumanMessage('You are playing a collaborative word gue') | AIMessage('') | ToolMessage("Noted: Target word is 'exactly'. Related") | AIMessage('CLUE: Without any deviation or error.')
  [obs:player_1] 'You are playing a collaborative word guessing game in which you have to guess a target word that ano'
  [memory:guesser] 4 messages: HumanMessage('You are playing a collaborative word gue') | AIMessage('') | ToolMessage("Noted: First clue: 'Without any deviatio") | AIMessage('GUESS: exact')
  [obs:player_0] 'GUESS: exact'
  [memory:describer] 6 messages: HumanMessage('You are playing a collaborative word gue') | AIMessage('') | ToolMessage("Noted: Target word is 'exactly'. Related") | AIMessage('CLUE: Without any deviation or error.') | HumanMessage('GUESS: exact\n\nSTRATEGIES AND OBSERVATION') | AIMessage('CLUE: With no room for error.'

In [321]:
# Display the last episode's steps
last_episode = all_episodes_data[-1]
print(f"Last episode: {len(last_episode)} steps")
print("-" * 60)
for idx, (agent_id, context, response, reward) in enumerate(last_episode):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"  Agent({agent_id}) <- Context: {context}")
    print(f"  Agent({agent_id}) -> Response: {response}")
    print("-" * 60)

Last episode: 8 steps
------------------------------------------------------------
Step 0 / Reward 0.00:
  Agent(player_0) <- Context: {'role': 'user', 'content': 'You are playing a collaborative word guessing game in which you have to describe a target word for another player to guess.\n\nRules:\n(a) You have to reply in the form: CLUE: <some text>. Guesses from the other player will start with GUESS.\n(b) You cannot use the target word itself, parts or morphological variants of it in your description.\n(c) In addition, the same rules apply for related words which are provided below.\n\nEnd conditions:\n(i) If you use the target word or a related word in your description, then you lose.\n(ii) If the other player can guess the target word in 3 tries, you both win.\n\nLet us start.\n\nThis is the target word that you need to describe and that the other player needs to guess:\n\nredirect\n\nRelated words are:\n\n- reroute\n- forward\n- divert\n\nImportant: You are under time pressure, gi

## 5. After the game

In [322]:
results_dir = callbacks.callbacks[0].results_folder.results_dir_path
run_dir     = callbacks.callbacks[0].results_folder.run_dir
print(f"Results saved to: {results_dir}")
print(f"Run dir:          {run_dir}")
print()
print("To score results:")
print(f"  clem score -g {GAME} -r playpen-records") #or -r PATH_TO_FOLDER
print(f"  clem eval -r playpen-records")   #or -r PATH_TO_FOLDER

Results saved to: playpen-records
Run dir:          infinite

To score results:
  clem score -g taboo -r playpen-records
  clem eval -r playpen-records


## 6. For debugging (this will enable the content of every LLM call)

In [323]:
"""
from langchain_core.callbacks import BaseCallbackHandler

class MessageSpy(BaseCallbackHandler):
      def on_chat_model_start(self, serialized, messages, **kwargs):
          print("\n=== MESSAGES TO LLM ===")
          for msg in messages[0]:
              print(f"  {type(msg).__name__}: {msg.content}")
          print("======================\n")
"""

# AND add this line to the act method:
#                bla bla recursion limit = 100,
#                "callbacks": [MessageSpy()],

'\nfrom langchain_core.callbacks import BaseCallbackHandler\n\nclass MessageSpy(BaseCallbackHandler):\n      def on_chat_model_start(self, serialized, messages, **kwargs):\n          print("\n=== MESSAGES TO LLM ===")\n          for msg in messages[0]:\n              print(f"  {type(msg).__name__}: {msg.content}")\n          print("======================\n")\n'